In [79]:
import pandas as pd
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
import tensorflow as tf
import pickle
from sklearn.model_selection import train_test_split    

data = pd.read_csv("D:\\Teja\\python_Automation\\pythonProject\\ANN_Regression_Data\\regression_sample.csv")
print(data.info())
print(data.head())


input=data.drop('price',axis=1)
target=data['price']

input.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   area_sqft              200 non-null    int64 
 1   bedrooms               200 non-null    int64 
 2   bathrooms              200 non-null    int64 
 3   age_years              200 non-null    int64 
 4   distance_from_city_km  200 non-null    int64 
 5   parking                200 non-null    int64 
 6   furnishing             200 non-null    object
 7   location               200 non-null    object
 8   price                  200 non-null    int64 
dtypes: int64(7), object(2)
memory usage: 14.2+ KB
None
   area_sqft  bedrooms  bathrooms  age_years  distance_from_city_km  parking  \
0       1360         1          1         17                     22        0   
1       4272         1          1          7                     30        2   
2       3592         3          2      

,area_sqft,bedrooms,bathrooms,age_years,distance_from_city_km,parking,furnishing,location
0,1360,1,1,17,22,0,Furnished,Urban
1,4272,1,1,7,30,2,Furnished,Suburban
2,3592,3,2,13,17,1,Furnished,Urban
3,966,3,2,9,26,0,Unfurnished,Rural
4,4926,3,3,27,36,1,Furnished,Suburban


In [63]:
onehot_encoder = OneHotEncoder(handle_unknown='ignore')
input_encoded = onehot_encoder.fit_transform(input[['furnishing', 'location']])
encoded=pd.DataFrame(input_encoded.toarray(), columns=onehot_encoder.get_feature_names_out(['furnishing', 'location']))
final = pd.concat([input.drop(['furnishing', 'location'], axis=1), encoded], axis=1)
final.head()

,area_sqft,bedrooms,bathrooms,age_years,distance_from_city_km,parking,furnishing_Furnished,furnishing_Semi-Furnished,furnishing_Unfurnished,location_Rural,location_Suburban,location_Urban
0,1360,1,1,17,22,0,1.0,0.0,0.0,0.0,0.0,1.0
1,4272,1,1,7,30,2,1.0,0.0,0.0,0.0,1.0,0.0
2,3592,3,2,13,17,1,1.0,0.0,0.0,0.0,0.0,1.0
3,966,3,2,9,26,0,0.0,0.0,1.0,1.0,0.0,0.0
4,4926,3,3,27,36,1,1.0,0.0,0.0,0.0,1.0,0.0


In [64]:
import pickle

# Save the encoders
with open('one_hot_encoder.pkl', 'wb') as f:
    pickle.dump(onehot_encoder, f)

print("Encoders saved successfully.")

Encoders saved successfully.


In [65]:
x_train, x_test, y_train, y_test = train_test_split(final, target, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled  = scaler.transform(x_test)

# Scale target (price) — important for regression with large values
target_scaler = StandardScaler()
y_train_scaled = target_scaler.fit_transform(y_train.values.reshape(-1, 1))
y_test_scaled  = target_scaler.transform(y_test.values.reshape(-1, 1))

print("x_train shape:", x_train_scaled.shape)
print("y_train scaled sample:", y_train_scaled[:3])


x_train shape: (160, 12)
y_train scaled sample: [[-0.33056708]
 [ 2.05799968]
 [-0.35241373]]


In [66]:
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('target_scaler.pkl', 'wb') as f:
    pickle.dump(target_scaler, f)

print("Scalers saved.")


Scalers saved.


In [67]:
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout

# Simple model — best for small datasets (30 rows)
model = Sequential([
    Dense(32, activation='relu', input_shape=(x_train_scaled.shape[1],)),
    Dropout(0.1),
    Dense(16, activation='relu'),
    Dense(1)
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='mean_squared_error',
              metrics=['mean_absolute_error'])
model.summary()


d:\Teja\python_Automation\pythonProject\Teja_repo\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_19 (Dense)                │ (None, 32)             │           416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 961 (3.75 KB)

 Trainable params: 961 (3.75 KB)

 Non-trainable params: 0 (0.00 B)

In [68]:
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
early_stopping       = EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True)
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)


In [69]:
history = model.fit(x_train_scaled, y_train_scaled,
                    validation_data=(x_test_scaled, y_test_scaled),
                    epochs=100, batch_size=4,
                    callbacks=[early_stopping, tensorboard_callback],
                    verbose=1)
model.save('ann_regression_model.keras')
print("Model saved.")


Epoch 1/100


40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - loss: 1.1495 - mean_absolute_error: 0.8875 - val_loss: 0.9471 - val_mean_absolute_error: 0.8184
Epoch 2/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.8818 - mean_absolute_error: 0.7626 - val_loss: 0.7457 - val_mean_absolute_error: 0.7125
Epoch 3/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.7502 - mean_absolute_error: 0.7106 - val_loss: 0.5886 - val_mean_absolute_error: 0.6259
Epoch 4/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.5789 - mean_absolute_error: 0.6035 - val_loss: 0.4410 - val_mean_absolute_error: 0.5323
Epoch 5/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.3958 - mean_absolute_error: 0.4903 - val_loss: 0.3033 - val_mean_absolute_error: 0.4168
Epoch 6/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.2986 - mean_absolute_error: 0.4226 - val_loss: 0.1857 - val_mean_absolute_error: 0.3159
Epoch 7/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.2445 - mean_absolute_error: 0.3669 - val_loss: 0.1295 

In [71]:
#load tensorboard extension
%load_ext tensorboard
%tensorboard --logdir logs/fit

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6008 (pid 24156), started 0:03:15 ago. (Use '!kill 24156' to kill it.)

In [ ]:
# load the pickle


with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
with open('target_scaler.pkl', 'rb') as f:
    target_scaler = pickle.load(f)
    


In [73]:
#Eveluate the model on test data
test_loss, test_mae = model.evaluate(x_test_scaled, y_test_scaled, verbose=0)
print(f"Test Loss: {test_loss:.4f}, Test MAE: {test_mae:.4f}")

Test Loss: 0.0081, Test MAE: 0.0699


In [ ]:
import pandas as pd

# Input new house data for prediction
input_data = {
    'area_sqft':             [1500, 3200],
    'bedrooms':              [3,    5],
    'bathrooms':             [2,    4],
    'age_years':             [5,    2],
    'distance_from_city_km': [10,   3],
    'parking':               [1,    2],
    'furnishing':            ['Furnished', 'Semi-Furnished'],
    'location':              ['Suburban',  'Urban']
}

input_df = pd.DataFrame(input_data)

# Step 1: One-hot encode furnishing and location
enc = onehot_encoder.transform(input_df[['furnishing', 'location']])
enc_df = pd.DataFrame(enc.toarray(), columns=onehot_encoder.get_feature_names_out(['furnishing', 'location']))
input_final = pd.concat([input_df.drop(['furnishing', 'location'], axis=1), enc_df], axis=1)

# Step 2: Scale features
input_scaled = scaler.transform(input_final)

# Step 3: Predict
pred_scaled = model.predict(input_scaled)

# Step 4: Inverse transform to get actual rupee price
pred_actual = target_scaler.inverse_transform(pred_scaled)

print("=" * 50)
for i in range(len(pred_actual)):
    print(f"House {i+1}:")
    print(f"  Area       : {input_data['area_sqft'][i]} sqft")
    print(f"  Location   : {input_data['location'][i]}")
    print(f"  Furnishing : {input_data['furnishing'][i]}")
    print(f"  Predicted  : ₹{pred_actual[i][0]:,.0f}")
    print(f"  Scaled out : {pred_scaled[i][0]:.4f}")
print("=" * 50)
print(f"\nTarget scaler mean : ₹{target_scaler.mean_[0]:,.0f}")
print(f"Target scaler std  : ₹{target_scaler.scale_[0]:,.0f}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 188ms/step
Predicted Prices: [6.6164345e+06 1.8754382e+07]
